In [41]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import AgentExecutor, create_react_agent
from langsmith import Client
import requests
import os
from  dotenv import load_dotenv

load_dotenv()

True

### Making of object for search tool which will do searching in browser. (it is builtin tool)

In [42]:
search_tool = DuckDuckGoSearchRun()

# print(search_tool.invoke("Latest newrs of today"))

### Making of Whether Tool (Custom Tool)

In [43]:
@tool
def get_weather_data(city: str) -> str:
    """
    This function fetches the current weather data for a given city.
    """
    url = f"https://api.weatherstack.com/current?access_key={os.getenv("WEATHER_STACK_API")}&query={city}"

    response = requests.get(url)

    return response.json()

### Creatng llm

In [44]:
llm = ChatGoogleGenerativeAI(
	model="gemini-3.8-flash",
	google_api_key=os.getenv("GOOGLE_API_KEY"),
	temperature=0.3,
#	max_tokens=20,
)

# print(llm.invoke("Hii"))

### Prompt making

In [45]:
client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

### Making of react - Reasoning and Acting agent

In [46]:
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

### Making of AgentExecuter

In [47]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True
)

### Difference between Agent and AgentExecutor
The Agent is the brain that thinks and decides what to do, while the Agent Executor is the body (or runtime environment) that actually executes those decisions and manages the loop.

In [48]:
result = agent_executor.invoke({
    "input": "Find the capital of Madhya Pradesh and tell about its weatheer condition also i want to go from delhi to goa so give me diff ways to go"
})
print(result["output"])



> Entering new AgentExecutor chain...
Action: get_weather_data
Action Input: Bhopal{'request': {'type': 'City', 'query': 'Bhopal, India', 'language': 'en', 'unit': 'm'}, 'location': {'name': 'Bhopal', 'country': 'India', 'region': 'Madhya Pradesh', 'lat': '23.267', 'lon': '77.400', 'timezone_id': 'Asia/Kolkata', 'localtime': '2026-10-25 13:56', 'localtime_epoch': 1792936560, 'utc_offset': '5.50'}, 'current': {'observation_time': '08:26 AM', 'temperature': 32, 'weather_code': 113, 'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0001_sunny.png'], 'weather_descriptions': ['Sunny'], 'astro': {'sunrise': '06:22 AM', 'sunset': '05:47 PM', 'moonrise': '05:01 PM', 'moonset': '05:16 AM', 'moon_phase': 'Waxing Gibbous', 'moon_illumination': 98}, 'air_quality': {'co': '204', 'no2': '0.2', 'o3': '146', 'so2': '5.4', 'pm2_5': '19.2', 'pm10': '33.5', 'us-epa-index': '2', 'gb-defra-index': '2'}, 'wind_speed': 4, 'wind_degree': 170, 'wind_dir': 'S', 'pressure':